# Vintage matrices and the anatomy of revisions

Every cell of a vintage matrix is a number as it was actually published on a given date. This notebook loads the four matrices used in the project (CPI and PPI monthly, PCE prices quarterly, from the Federal Reserve Bank of Philadelphia's Real-Time Data Set for Macroeconomists; retail sales monthly, rebuilt from ALFRED intervals for series RSAFS) and measures how first releases are later revised.

In [1]:
import sys
sys.path.insert(0, "../src")
import pandas as pd
from realtime_forecast.vintages import load_alfred_matrix, load_rtdsm_matrix
from realtime_forecast.revisions import (
    growth_revision_frame, revision_stats, revision_stats_by_phase,
    noise_vs_news, vintage_coverage,
)
pd.set_option("display.width", 120)


## Coverage, exactly as retrieved

In [2]:
matrices = {
    'cpi': load_rtdsm_matrix('../data/raw/pcpiMvMd.xlsx', 'M'),
    'ppi': load_rtdsm_matrix('../data/raw/pppiMvMd.xlsx', 'M'),
    'retail': load_alfred_matrix('../data/raw/alfred_rsafs_vintages.csv', 'M'),
    'pce': load_rtdsm_matrix('../data/raw/pconQvQd.xlsx', 'Q'),
}
pd.DataFrame({k: vintage_coverage(m) for k, m in matrices.items()}).T

,n_observations,n_vintages,first_vintage,last_vintage,first_obs_in_first_vintage,last_obs_in_first_vintage,first_obs_in_latest_vintage,last_obs_in_latest_vintage,cells_populated
cpi,957,336,1998-11,2026-10,1948-01,1998-10,1947-01,2026-09,265056
ppi,957,336,1998-11,2026-10,1947-04,1998-10,1947-04,2026-09,264257
retail,417,305,2001-06,2026-10,1992-01,2001-05,1992-01,2026-09,80816
pce,319,245,1965Q4,2026Q4,1947Q1,1965Q2,1947Q1,2026Q3,47572


## Revision statistics on the growth scale

Revisions are measured on growth rates (100 x log difference), maturity-restricted: an observation counts only if its first vintage is at least 24 vintages (monthly) or 8 vintages (quarterly) before the final vintage, so its latest value is close to settled.

In [3]:
frames = {
    'cpi': growth_revision_frame(matrices['cpi'], 24),
    'ppi': growth_revision_frame(matrices['ppi'], 24),
    'retail': growth_revision_frame(matrices['retail'], 24),
    'pce': growth_revision_frame(matrices['pce'], 8),
}
pd.DataFrame({k: revision_stats(f) for k, f in frames.items()}).T

,n,mean_revision,mean_abs_revision,median_abs_revision,sd_revision,share_revised_up,share_revised_down,share_unrevised,p90_abs_revision,corr_first_revision
cpi,932.0,0.000583,0.031288,0.000000,0.061650,0.290773,0.189914,0.519313,0.109987,-0.170145
ppi,929.0,0.001263,0.080298,0.000000,0.178277,0.212056,0.182992,0.604952,0.281262,-0.361509
retail,392.0,0.025800,0.392928,0.315877,0.525407,0.561224,0.438776,0.000000,0.835107,-0.082064
pce,310.0,0.016089,0.133094,0.103656,0.177167,0.551613,0.448387,0.000000,0.303542,-0.143496


## Revisions by business-cycle phase (NBER)

In [4]:
for key, frame in frames.items():
    freq = 'Q' if key == 'pce' else 'M'
    print(key)
    print(pd.DataFrame(revision_stats_by_phase(frame, freq=freq)).T[['n', 'mean_revision', 'mean_abs_revision', 'reported']])

cpi
               n mean_revision mean_abs_revision reported
expansion    901      0.000266          0.030344     True
contraction   31      0.009778           0.05872     True
ppi
               n mean_revision mean_abs_revision reported
expansion    898      0.001556            0.0744     True
contraction   31     -0.007225          0.251139     True
retail
               n mean_revision mean_abs_revision reported
expansion    361      0.027784          0.377083     True
contraction   31      0.002694          0.577447     True
pce
               n mean_revision mean_abs_revision reported
expansion    300      0.019526           0.13282     True
contraction   10      -0.08704          0.141306    False


## Noise or news?

Regressing the revision on the first release: under a pure-news process the slope is zero. A nonzero slope means part of the first release was predictable error (noise) at release time.

In [5]:
pd.DataFrame({
    'cpi': noise_vs_news(frames['cpi'], 12),
    'ppi': noise_vs_news(frames['ppi'], 12),
    'retail': noise_vs_news(frames['retail'], 12),
    'pce': noise_vs_news(frames['pce'], 4),
}).T

,n,intercept,slope_on_first_release,slope_se_hac,slope_t,slope_p,r_squared
cpi,932.0,0.009279,-0.030253,0.008994,-3.363565,0.000769,0.028949
ppi,929.0,0.025309,-0.097982,0.020446,-4.792239,0.000002,0.130689
retail,392.0,0.034468,-0.025262,0.017189,-1.469636,0.141660,0.006734
pce,310.0,0.044944,-0.038520,0.019816,-1.943848,0.051914,0.020591


## The revision-risk band

The 90th percentile of the absolute revision of first-release growth is the band a decision-maker should attach to a release: nine times out of ten, the settled figure lands within this distance of the number first printed.

In [6]:
{k: round(float(f['revision_latest_minus_first'].abs().quantile(0.90)), 3)
 for k, f in frames.items()}

{'cpi': 0.11, 'ppi': 0.281, 'retail': 0.835, 'pce': 0.304}